# End-to-end Docking Notebook (self-contained)
Implements a minimal DiffDock-style score model, sampling loop, AutoDock Vina affinity estimation, and PDB2PQR-based protonation workflow in one place for Kaggle GPU runs.

In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/DiffDock'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/gcorso/DiffDock.git', REPO_DIR], check=True)
print('Repo at', REPO_DIR)
os.chdir(REPO_DIR)


In [ ]:
# Create conda env (no activate). Install all deps into it.
!conda env remove -n diffdock -y >/dev/null 2>&1 || true
!conda create -y -n diffdock python=3.10 >/dev/null
!conda run -n diffdock conda install -y -c pytorch -c nvidia -c conda-forge \
    pytorch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 pytorch-cuda=11.8 \
    rdkit biopython pandas tqdm pyyaml scipy einops requests openbabel pdb2pqr propka >/dev/null
!conda run -n diffdock pip install -q torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv pyg_lib -f https://data.pyg.org/whl/torch-2.2.0+cu118.html -f https://download.pytorch.org/whl/cu118
!conda run -n diffdock pip install -q e3nn autodock-vina meeko
!conda run -n diffdock python - <<'PY'
import sys, torch
print('Conda env ready:', sys.prefix)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
PY

In [ ]:
import os, json, math, tempfile, subprocess, textwrap, tarfile, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import radius_graph
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolfiles import MolToMolBlock
from rdkit.Chem.rdMolAlign import AlignMol
from rdkit.Chem.Draw import IPythonConsole
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
import sys, os, site
ENV_PREFIX = '/opt/conda/envs/diffdock'
site_pkgs = os.path.join(ENV_PREFIX, 'lib', 'python3.10', 'site-packages')
if site_pkgs not in sys.path:
    sys.path.insert(0, site_pkgs)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Using env site-packages:', site_pkgs)
print('Repo on sys.path:', REPO_DIR)


## Load default inference args
Reads `default_inference_args.yaml` to reuse the same sampling hyperparameters for runs.

In [ ]:
import yaml, os
inference_args_path = os.path.join(REPO_DIR, 'default_inference_args.yaml')
with open(inference_args_path) as f:
    default_inference_args = yaml.safe_load(f)
print('Loaded default inference args keys:', list(default_inference_args.keys()))


## Build full DiffDock architecture (random init)
Constructs the official AAModel/CGModel via `get_model` with reasonable default hyperparameters and random weights (no pretrained files required).

In [ ]:
import yaml
from argparse import Namespace
from functools import partial
from utils.utils import get_model
from utils.diffusion_utils import t_to_sigma as t_to_sigma_compl

# Choose whether to use pretrained weights (if downloaded) or random init
use_pretrained = True
MODEL_DIR = os.path.join(REPO_DIR, 'models/diffdock_models')
CKPT = 'best_ema_inference_epoch_model.pt'
params_path = os.path.join(MODEL_DIR, 'model_parameters.yml')

if use_pretrained and os.path.exists(params_path):
    with open(params_path) as f:
        model_args = Namespace(**yaml.full_load(f))
    state_path = os.path.join(MODEL_DIR, CKPT)
else:
    # fallback minimal defaults
    model_args = Namespace(
        all_atoms=True,
        no_torsion=False,
        num_conv_layers=2,
        max_radius=5.0,
        scale_by_sigma=True,
        sigma_embed_dim=32,
        norm_by_sigma=True,
        ns=16,
        nv=4,
        distance_embed_dim=32,
        cross_distance_embed_dim=32,
        no_batch_norm=False,
        use_second_order_repr=False,
        cross_max_distance=80.0,
        dynamic_max_cross=False,
        smooth_edges=False,
        odd_parity=False,
        embedding_type='sinusoidal',
        embedding_scale=10000,
        dropout=0.0,
        affinity_prediction=False,
        parallel=1,
        rmsd_classification_cutoff=[2.0],
        atom_rmsd_classification_cutoff=[2.0],
        parallel_aggregators="mean max min std",
        not_fixed_center_conv=False,
        no_aminoacid_identities=False,
        include_miscellaneous_atoms=False,
        sh_lmax=2,
        no_differentiate_convolutions=False,
        tp_weights_layers=2,
        num_prot_emb_layers=0,
        reduce_pseudoscalars=False,
        embed_also_ligand=False,
        atom_confidence_loss_weight=0.0,
        depthwise_convolution=False,
        crop_beyond=None,
        esm_embeddings_path=None,
    )
    state_path = None

model_args.norm_by_sigma = True

t_to_sigma = partial(t_to_sigma_compl, args=model_args)
model = get_model(model_args, device=device, t_to_sigma=t_to_sigma, no_parallel=True)
if state_path and os.path.exists(state_path):
    state = torch.load(state_path, map_location='cpu')
    model.load_state_dict(state, strict=True)
    print('Loaded pretrained weights from', state_path)
else:
    print('Using random initialization (no pretrained weights found).')
model = model.to(device)
model.eval()


## Download pretrained weights (optional but recommended)
Fetches official DiffDock model zip from releases and places it under `models/diffdock_models`. Skip if you prefer random initialization.

In [ ]:
import zipfile, urllib.request
MODEL_DIR = os.path.join(REPO_DIR, 'models/diffdock_models')
os.makedirs(MODEL_DIR, exist_ok=True)
zip_path = os.path.join(REPO_DIR, 'models', 'diffdock_models.zip')
if not any(Path(MODEL_DIR).glob('*.pt')):
    urls = [
        'https://github.com/gcorso/DiffDock/releases/latest/download/diffdock_models.zip',
        'https://github.com/gcorso/DiffDock/releases/download/v1.1/diffdock_models.zip'
    ]
    downloaded = False
    for u in urls:
        try:
            print('Downloading', u)
            urllib.request.urlretrieve(u, zip_path)
            downloaded = True
            break
        except Exception as e:
            print('Failed', u, e)
    if not downloaded:
        raise RuntimeError('Could not download pretrained weights.')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(os.path.join(REPO_DIR, 'models'))
    print('Extracted weights to', MODEL_DIR)
else:
    print('Weights already present in', MODEL_DIR)


## Use fixed sample paths from repo
Hardcode the included example: protein `data/1a0q/1a0q_protein_processed.pdb` and ligand `data/1a0q/1a0q_ligand.sdf`. If missing (e.g., on fresh Kaggle), we download 6w70 and build a ligand from SMILES.

In [ ]:
import os, subprocess, urllib.request
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem

protein_path = os.path.join(REPO_DIR, 'data/1a0q/1a0q_protein_processed.pdb')
ligand_sdf = os.path.join(REPO_DIR, 'data/1a0q/1a0q_ligand.sdf')
fallback_smiles = 'COc1ccc(cc1)n2c3c(c(n2)C(=O)N)CCN(C3=O)c4ccc(cc4)N5CCCCC5=O'

def fetch_pdb(pdb_id='6w70', out_path='protein.pdb'):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    urllib.request.urlretrieve(url, out_path)
    return out_path

def build_ligand_from_sdf(path):
    supp = Chem.SDMolSupplier(path, removeHs=False)
    mol = next((m for m in supp if m), None)
    if mol is None:
        raise ValueError(f'No molecule in {path}')
    return mol

def build_ligand_from_smiles(smiles, out_sdf='ligand.sdf', n_confs=10):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3(); params.numThreads = 0; params.pruneRmsThresh = 0.5
    AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=0)
    w = Chem.SDWriter(out_sdf)
    for cid in range(mol.GetNumConformers()):
        w.write(mol, confId=cid)
    w.close()
    return mol

if Path(protein_path).exists():
    print('Using protein:', protein_path)
else:
    protein_path = fetch_pdb(out_path=os.path.join(REPO_DIR, 'protein.pdb'))
    print('Fallback download protein:', protein_path)

if Path(ligand_sdf).exists():
    ligand = build_ligand_from_sdf(ligand_sdf)
    print('Using ligand:', ligand_sdf, 'confs:', ligand.GetNumConformers())
else:
    ligand = build_ligand_from_smiles(fallback_smiles, out_sdf=os.path.join(REPO_DIR, 'ligand.sdf'))
    ligand_sdf = os.path.join(REPO_DIR, 'ligand.sdf')
    Chem.SDWriter(ligand_sdf).write(ligand)
    print('Fallback ligand built to', ligand_sdf)


## Build lightweight DiffDock-style score model
We implement a small SE(3)-aware graph network (radial features + vector outputs) for score matching on ligand atoms.

In [ ]:
class RadialEmbedding(nn.Module):
    def __init__(self, n_rbf=32, cutoff=6.0):
        super().__init__(); self.cutoff=cutoff
        centers = torch.linspace(0, cutoff, n_rbf)
        self.register_buffer('centers', centers)
        self.gamma = 10.0
    def forward(self, distances):
        x = distances.unsqueeze(-1) - self.centers
        return torch.exp(-self.gamma * x * x)

class ScoreModel(MessagePassing):
    def __init__(self, node_dim=64, edge_dim=64, cutoff=6.0):
        super().__init__(aggr='add')
        self.rbf = RadialEmbedding(32, cutoff)
        self.node_mlp = nn.Sequential(nn.Linear(6, node_dim), nn.SiLU(), nn.Linear(node_dim, node_dim))
        self.edge_mlp = nn.Sequential(nn.Linear(32, edge_dim), nn.SiLU(), nn.Linear(edge_dim, edge_dim))
        self.msg_mlp = nn.Sequential(nn.Linear(node_dim*2 + edge_dim, node_dim), nn.SiLU(), nn.Linear(node_dim, node_dim))
        self.out_mlp = nn.Sequential(nn.Linear(node_dim, node_dim), nn.SiLU(), nn.Linear(node_dim, 3))
    def forward(self, data):
        pos, z = data.pos, data.z.float().unsqueeze(-1)
        h = self.node_mlp(torch.cat([z/100., torch.zeros_like(z), torch.ones_like(z), torch.zeros_like(z), torch.zeros_like(z), torch.zeros_like(z)], dim=-1))
        edge_index = radius_graph(pos, r=6.0, loop=False)
        row, col = edge_index
        rel = pos[row] - pos[col]
        dist = rel.norm(dim=-1) + 1e-6
        e = self.edge_mlp(self.rbf(dist))
        m = torch.cat([h[row], h[col], e], dim=-1)
        m = self.msg_mlp(m)
        agg = torch.zeros_like(h); agg.index_add_(0, row, m)
        h = h + agg
        score = self.out_mlp(h)
        return score, edge_index

def mol_to_graph(mol, conf_id=0):
    conf = mol.GetConformer(conf_id)
    pos = torch.tensor(conf.GetPositions(), dtype=torch.float)
    z = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.long)
    return Data(pos=pos, z=z)

model = ScoreModel().to(device)
optim = torch.optim.Adam(model.parameters(), lr=3e-4)

def score_matching_step(data):
    model.train(); data = data.to(device)
    noise = torch.randn_like(data.pos)
    noisy = data.pos + 0.5 * noise
    data.pos = noisy
    pred, _ = model(data)
    loss = ((pred + noise) ** 2).mean()
    optim.zero_grad(); loss.backward(); optim.step()
    return loss.item()

## Quick train on ligand pose (toy score matching)
Trains on a single conformer to predict added noise; this is illustrative, not production DiffDock.

In [ ]:
base_graph = mol_to_graph(ligand, 0)
for step in range(200):
    loss = score_matching_step(base_graph.clone())
    if step % 50 == 0: print('step', step, 'loss', loss)

## Sample poses with reverse diffusion and rank by score norm
Adds noise, denoises for a few steps, and ranks by predicted score magnitude (lower is better).

In [ ]:
def sample_pose(data, steps=30, sigma=1.0):
    data = data.to(device)
    pos = data.pos + sigma * torch.randn_like(data.pos)
    for t in reversed(range(steps)):
        data.pos = pos
        with torch.no_grad(): score, _ = model(data)
        pos = pos + 0.05 * score
    with torch.no_grad():
        data.pos = pos; score, _ = model(data); s_norm = score.norm(dim=-1).mean().item()
    return pos.cpu().numpy(), s_norm

samples = []
for i in range(8):
    p, s = sample_pose(base_graph.clone())
    samples.append((p, s))
samples = sorted(samples, key=lambda x: x[1])
print('Best score norm', samples[0][1])
top_pos = samples[0][0]

## Run official DiffDock inference (pretrained)
Optional: call the repo's inference pipeline on the chosen protein/ligand to generate and rank poses with the full model.

In [ ]:
import pandas as pd, uuid, subprocess, sys

tmp_csv = os.path.join(REPO_DIR, 'notebook_inference.csv')
pd.DataFrame([
    {
        'complex_name': f'nb_{uuid.uuid4().hex[:8]}',
        'protein_path': protein_path,
        'ligand_description': ligand_sdf,
        'protein_sequence': ''
    }
]).to_csv(tmp_csv, index=False)

print('Running DiffDock inference on', protein_path, ligand_sdf)
cmd = ['conda', 'run', '-n', 'diffdock', 'python', '-m', 'inference', '--config', 'default_inference_args.yaml',
       '--protein_ligand_csv', tmp_csv, '--out_dir', os.path.join(REPO_DIR, 'results/notebook_diffdock'),
       '--samples_per_complex', '5', '--batch_size', '1']
subprocess.run(cmd, check=False, cwd=REPO_DIR)
print('Done; outputs in results/notebook_diffdock')

## Save top pose to SDF
Writes the best-scoring pose to disk.

In [ ]:
best = Chem.Mol(ligand)
conf = best.GetConformer(0)
for i in range(best.GetNumAtoms()):
    x,y,z = top_pos[i]
    conf.SetAtomPosition(i, (float(x), float(y), float(z)))
Chem.SDWriter('best_diffusion_pose.sdf').write(best)
print('Saved best_diffusion_pose.sdf')

## AutoDock Vina affinity estimation
Prepares pdbqt via Meeko and runs Vina centered on the ligand bbox.

In [ ]:
from meeko import MoleculePreparation, PDBQTWriterLegacy
from autodock_vina import Vina

def prepare_receptor_pdbqt(pdb_file, out_file='receptor.pdbqt'):
    prep = MoleculePreparation()
    struct = prep.prepare_from_pdb_file(pdb_file)
    writer = PDBQTWriterLegacy()
    writer.write(struct, out_file)
    return out_file

def prepare_ligand_pdbqt(sdf_in='best_diffusion_pose.sdf', out_file='ligand.pdbqt'):
    prep = MoleculePreparation()
    struct = prep.prepare_from_file(sdf_in)
    writer = PDBQTWriterLegacy()
    writer.write(struct, out_file)
    return out_file

rec_pdbqt = prepare_receptor_pdbqt(protein_path)
lig_pdbqt = prepare_ligand_pdbqt()

def vina_box_from_ligand(mol):
    conf = mol.GetConformer(0)
    coords = np.array(conf.GetPositions())
    center = coords.mean(0)
    size = coords.max(0) - coords.min(0) + 10.0
    return center.tolist(), size.tolist()

center, size = vina_box_from_ligand(ligand)
v = Vina(sf_name='vina', seed=42)
v.set_receptor(rec_pdbqt)
v.set_ligand_from_file(lig_pdbqt)
v.compute_vina_maps(center=center, box_size=size)
scores = v.score()
print('Vina first pose score (kcal/mol):', scores[0])
v.dock(exhaustiveness=8, n_poses=5)
v.write_poses('vina_out.pdbqt', n_poses=5)
print('Saved vina_out.pdbqt')

## Protonation with PDB2PQR then Vina
Runs pdb2pqr (propka) to assign protonation, converts to pdbqt, reruns Vina.

In [ ]:
from pdb2pqr.main import main as pdb2pqr_main

pdb2pqr_out = 'protein_protonated.pqr'
pdb2pqr_main(['--ff=PARSE', protein_path, pdb2pqr_out])
# Convert PQR back to PDB using openbabel, then to PDBQT
subprocess.run(['obabel', pdb2pqr_out, '-O', 'protein_protonated.pdb'], check=True)
rec_pdbqt_prot = prepare_receptor_pdbqt('protein_protonated.pdb', out_file='receptor_protonated.pdbqt')
v2 = Vina(sf_name='vina', seed=7)
v2.set_receptor(rec_pdbqt_prot)
v2.set_ligand_from_file(lig_pdbqt)
v2.compute_vina_maps(center=center, box_size=size)
v2.dock(exhaustiveness=8, n_poses=3)
v2.write_poses('vina_out_protonated.pdbqt', n_poses=3)
print('Saved vina_out_protonated.pdbqt')

### Notes
- The score model here is a minimal illustrative variant, not the full DiffDock architecture.
- Training uses a single pose for speed; extend to datasets for real use.
- Vina requires a good box; adjust `center/size` as needed.
- PDB2PQR step needs openbabel; installed above.